# Homework 3: Machine Learning for Classification

ML Zoomcamp 2026 — reproducible solution using the pinned official dataset.

## Setup and data

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mutual_info_score

DATA_URL = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
df = pd.read_csv(DATA_URL)
target = 'converted'
X = df.drop(columns=[target]).copy()
categorical = X.select_dtypes(include=['object']).columns.tolist()
numerical = X.select_dtypes(exclude=['object']).columns.tolist()
X[categorical] = X[categorical].fillna('NA')
X[numerical] = X[numerical].fillna(0.0)

def fit_model(X_train, y_train, X_val, C=1.0):
    dv = DictVectorizer(sparse=False)
    train_matrix = dv.fit_transform(X_train.to_dict(orient='records'))
    val_matrix = dv.transform(X_val.to_dict(orient='records'))
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(train_matrix, y_train)
    return model, val_matrix


## Q1. Most frequent industry

In [2]:
industry_mode = df.industry.mode().iloc[0]
industry_mode

'technology'

**Answer:** `technology`.

## Q2. Highest-correlation pair

In [3]:
pairs = [('interaction_count','lead_score'), ('number_of_courses_viewed','lead_score'), ('number_of_courses_viewed','interaction_count'), ('annual_income','interaction_count')]
corr = X[numerical].corr().abs()
pair_scores = {pair: corr.loc[pair[0], pair[1]] for pair in pairs}
pair_scores

{('interaction_count', 'lead_score'): np.float64(0.9157457980418697), ('number_of_courses_viewed', 'lead_score'): np.float64(0.7572042392104886), ('number_of_courses_viewed', 'interaction_count'): np.float64(0.7216090038937035), ('annual_income', 'interaction_count'): np.float64(0.12284235135955959)}

**Answer:** `interaction_count` and `lead_score`.

## Split once for Q3–Q6

In [4]:
prepared = X.assign(converted=df.converted.to_numpy())
full_train, test = train_test_split(prepared, test_size=0.2, random_state=42)
train, val = train_test_split(full_train, test_size=0.25, random_state=42)
y_train = train.pop('converted'); y_val = val.pop('converted')

## Q3. Mutual information

In [5]:
mi_scores = {column: mutual_info_score(y_train, train[column]) for column in categorical}
mi_scores

{'lead_source': 0.03434605791617973, 'industry': 0.0019945636715935217, 'employment_status': 0.018965604906982875, 'location': 0.001042763579701525}

**Answer:** `lead_source`.

## Q4. Validation accuracy

In [6]:
model, val_matrix = fit_model(train, y_train, val)
original_accuracy = accuracy_score(y_val, model.predict(val_matrix))
original_accuracy

0.645

**Answer:** `0.65`.

## Q5. Feature elimination

In [7]:
differences = {}
for column in ['lead_source','number_of_courses_viewed','interaction_count']:
    reduced_model, reduced_matrix = fit_model(train.drop(columns=[column]), y_train, val.drop(columns=[column]))
    differences[column] = abs(original_accuracy - accuracy_score(y_val, reduced_model.predict(reduced_matrix)))
differences

{'lead_source': 0.0030000000000000027, 'number_of_courses_viewed': 0.0020000000000000018, 'interaction_count': 0.04400000000000004}

**Answer:** `number_of_courses_viewed`.

## Q6. Regularization

In [8]:
c_values = [1e-6, 1e-5, 1e-4, 1e-3]
c_scores = {}
for C in c_values:
    c_model, c_matrix = fit_model(train, y_train, val, C=C)
    c_scores[C] = accuracy_score(y_val, c_model.predict(c_matrix))
c_scores

{1e-06: 0.598, 1e-05: 0.598, 0.0001: 0.613, 0.001: 0.645}

**Answer:** `0.001`.

## Checks

In [9]:
assert industry_mode == 'technology'
assert max(pair_scores, key=pair_scores.get) == ('interaction_count','lead_score')
assert max(mi_scores, key=mi_scores.get) == 'lead_source'
assert round(original_accuracy,2) == 0.65
assert min(differences,key=differences.get) == 'number_of_courses_viewed'
assert max(c_values,key=lambda c:(c_scores[c],-c)) == 0.001
print('All Homework 3 checks passed.')

All Homework 3 checks passed.
